# Feature engineering experiments

This notebook documents the experiments that led to the final time- and frequency-domain feature set. It covers window extraction, dataset preparation, condition summaries, and challenge recordings.

> **Pipeline note:** Some cells intentionally preserve earlier iterations for comparison. The deduplicated implementation used by the command-line workflow lives in `src/machine_sentinel/features.py` and `src/machine_sentinel/datasets.py`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
SAMPLE_RATE = 200
WINDOW_SIZE = 200       # 1 second
COUNTS_PER_G = 4096.0   # MPU6050 at ±8g

In [ ]:
def band_power(frequencies, power, low, high):

    mask = (
        (frequencies >= low) &
        (frequencies < high)
    )

    return np.sum(power[mask])

In [ ]:
def extract_window_features(
    window_data,
    sample_rate=SAMPLE_RATE
):
    """
    Extract vibration features from one fixed-size window.
    """

    # ---------------------------------
    # 1. Raw counts -> acceleration (g)
    # ---------------------------------

    ax = window_data["ax"].to_numpy() / COUNTS_PER_G
    ay = window_data["ay"].to_numpy() / COUNTS_PER_G
    az = window_data["az"].to_numpy() / COUNTS_PER_G

    # ---------------------------------
    # 2. Remove DC / gravity
    # ---------------------------------

    ax_c = ax - np.mean(ax)
    ay_c = ay - np.mean(ay)
    az_c = az - np.mean(az)

    # ---------------------------------
    # 3. Time-domain features
    # ---------------------------------

    dynamic_mag = np.sqrt(
        ax_c**2 +
        ay_c**2 +
        az_c**2
    )

    # Overall 3-axis vibration strength
    rms = np.sqrt(
        np.mean(
            ax_c**2 +
            ay_c**2 +
            az_c**2
        )
    )

    # Spread of vector vibration magnitude
    ptp = np.ptp(dynamic_mag)

    # ---------------------------------
    # 4. FFT each axis separately
    # ---------------------------------

    fft_x = np.fft.rfft(ax_c)
    fft_y = np.fft.rfft(ay_c)
    fft_z = np.fft.rfft(az_c)

    frequencies = np.fft.rfftfreq(
        len(ax_c),
        d=1 / sample_rate
    )

    # ---------------------------------
    # 5. Combine spectral power
    # ---------------------------------

    power = (
        np.abs(fft_x) ** 2 +
        np.abs(fft_y) ** 2 +
        np.abs(fft_z) ** 2
    )

    # Ignore 0 Hz
    dominant_idx = np.argmax(power[1:]) + 1

    dominant_freq = frequencies[dominant_idx]

    # ---------------------------------
    # 6. Band energy
    # ---------------------------------

    energy_low = band_power(
        frequencies,
        power,
        1,
        30
    )

    energy_mid = band_power(
        frequencies,
        power,
        30,
        50
    )

    energy_high = band_power(
        frequencies,
        power,
        50,
        101
    )

    return {
        "rms": rms,
        "ptp": ptp,
        "dominant_freq": dominant_freq,
        "energy_low": energy_low,
        "energy_mid": energy_mid,
        "energy_high": energy_high
    }

In [ ]:
def extract_recording_features(df, recording_id, condition, label, window_size=WINDOW_SIZE):
    """
    Extract DSP features from a recording of accelerometer data.

    Parameters:
        df (pd.DataFrame):
            DataFrame containing accelerometer data for multiple recordings.
        recording_id (int):
            ID of the recording to extract features from.
        condition (str):
            Condition under which the recording was made.
        label (int):
            Label for the recording.
        window_size (int):
            Size of the window in samples.
        sample_rate (int):
            Sampling frequency in Hz.

    Returns:
        dict:
            Extracted features for the recording.
    """
    rows = []
    for start in range(0, len(df), window_size):
        end = start + window_size

        if end > len(df):
            break

        window_data = df.iloc[start:end]
        features = extract_window_features(window_data)
        rows.append({
            "recording_id": recording_id,
            "condition": condition,
            "label": label,
            "start_time_s": start / SAMPLE_RATE,
            "window" : start // window_size,
            **features
        })

    return pd.DataFrame(rows)

In [ ]:
df = pd.read_csv("../../data/raw/40hz.csv")

In [ ]:
features = extract_recording_features(df, recording_id="normal_40_01", condition="normal_40hz", label=0)

In [ ]:
features.head()

In [ ]:
# DATA 

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
SAMPLE_RATE = 200
WINDOW_SIZE = 200
COUNTS_PER_G = 4096.0

In [ ]:
def extract_recording_features(
    df,
    recording_id,
    condition,
    label,
    trim_seconds=1
):
    """
    Convert one complete recording into one feature row
    per 1-second window.
    """

    df = df.copy()

    # ---------------------------------------
    # Remove start/stop transient regions
    # ---------------------------------------

    trim_samples = int(trim_seconds * SAMPLE_RATE)

    if len(df) > 2 * trim_samples:
        df = df.iloc[
            trim_samples : len(df) - trim_samples
        ].reset_index(drop=True)

    rows = []

    # ---------------------------------------
    # Split recording into windows
    # ---------------------------------------

    for start in range(
        0,
        len(df),
        WINDOW_SIZE
    ):

        end = start + WINDOW_SIZE

        # Ignore incomplete final window
        if end > len(df):
            break

        window_data = df.iloc[start:end]

        features = extract_window_features(
            window_data,
            SAMPLE_RATE
        )

        rows.append({
            "recording_id": recording_id,
            "condition": condition,
            "label": label,

            "window": start // WINDOW_SIZE,
            "start_time_s": start / SAMPLE_RATE,

            "rms": features["rms"],
            "ptp": features["ptp"],
            "dominant_freq": features["dominant_freq"],

            "energy_low": features["energy_low"],
            "energy_mid": features["energy_mid"],
            "energy_high": features["energy_high"]
        })

    return pd.DataFrame(rows)

In [ ]:
DATA_DIR = Path("../../data/raw/final")
conditions = {
    "normal_40": {
        "folder": DATA_DIR / "normal_40",
        "label": 0
    },

    "loud_40": {
        "folder": DATA_DIR / "loud_40",
        "label": 1
    },

    "freq_60": {
        "folder": DATA_DIR / "freq_60",
        "label": 1
    }
}

In [ ]:
all_feature_tables = []

for condition, config in conditions.items():

    folder = config["folder"]
    label = config["label"]

    files = sorted(folder.glob("*.csv"))

    print(
        condition,
        "->",
        len(files),
        "files"
    )

    for file_path in files:

        print("Processing:", file_path.name)

        df = pd.read_csv(file_path)

        # Safety check
        required_columns = {
            "ax",
            "ay",
            "az"
        }

        if not required_columns.issubset(df.columns):
            print(
                "Skipping:",
                file_path.name,
                "- missing accelerometer columns"
            )
            continue

        feature_df = extract_recording_features(
            df=df,
            recording_id=file_path.stem,
            condition=condition,
            label=label,
            trim_seconds=1
        )

        all_feature_tables.append(
            feature_df
        )

In [ ]:
features_df = pd.concat(
    all_feature_tables,
    ignore_index=True
)

In [ ]:
print(features_df.shape, "feature rows extracted")

In [ ]:
print(features_df.head())

In [ ]:
print(
    features_df["condition"]
    .value_counts()
)

In [ ]:
print(
    features_df.groupby(
        ["condition", "recording_id"]
    ).size()
)

In [ ]:
processed_dir = DATA_DIR / "processed"

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
features_df.to_csv(
    processed_dir / "features.csv",
    index=False
)

In [ ]:
summary = (
    features_df
    .groupby("condition")
    [
        [
            "rms",
            "ptp",
            "dominant_freq",
            "energy_low",
            "energy_mid",
            "energy_high"
        ]
    ]
    .agg(["mean", "std"])
)

print(summary)

In [ ]:
plt.figure(figsize=(8, 6))

for condition in features_df["condition"].unique():

    subset = features_df[
        features_df["condition"] == condition
    ]

    plt.scatter(
        subset["dominant_freq"],
        subset["rms"],
        label=condition,
        alpha=0.7
    )

plt.xlabel("Dominant Frequency (Hz)")
plt.ylabel("3-axis RMS (g)")

plt.title(
    "Vibration Conditions in Feature Space"
)

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

for condition in features_df["condition"].unique():

    subset = features_df[
        features_df["condition"] == condition
    ]

    plt.scatter(
        subset["energy_mid"],
        subset["energy_high"],
        label=condition,
        alpha=0.7
    )

plt.xlabel("30–50 Hz Energy")
plt.ylabel("50–100 Hz Energy")

plt.title(
    "Frequency Band Energy by Condition"
)

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
print(
    features_df.groupby("condition")[
        "dominant_freq"
    ].value_counts()
)

In [ ]:
# Stationaery vs Tapping

def load_challenge_folder(folder_path, condition):

    tables = []

    folder = Path(folder_path)

    for file_path in sorted(folder.glob("*.csv")):

        df = pd.read_csv(file_path)

        feature_df = extract_recording_features(
            df=df,
            recording_id=file_path.stem,
            condition=condition,
            label=-1,          # unknown / challenge dataset
            trim_seconds=0
        )

        tables.append(feature_df)

    return pd.concat(
        tables,
        ignore_index=True
    )

In [ ]:
stationary_df = load_challenge_folder(
    DATA_DIR / "stationary",
    "stationary"
)

tapping_df = load_challenge_folder(
    DATA_DIR / "tapping",
    "tapping"
)

In [ ]:
challenge_df = pd.concat(
    [
        stationary_df,
        tapping_df
    ],
    ignore_index=True
)

In [ ]:
challenge_df.to_csv(
    processed_dir / "challenge.csv",
    index=False
)

In [ ]:
from pathlib import Path
import pandas as pd

def load_challenge_folder(folder_path, condition):
    folder = Path(folder_path)
    tables = []

    for file_path in sorted(folder.glob("*.csv")):
        print(f"Loading {file_path.name}...")
        df = pd.read_csv(file_path)

        feature_df = extract_recording_features(df, 
                                                recording_id=file_path.stem, 
                                                condition=condition,
                                                label=-1,
                                                trim_seconds=0)
        tables.append(feature_df)
    return pd.concat(tables, ignore_index=True)

In [ ]:
stationary_df = load_challenge_folder(
    DATA_DIR / "stationary",
    "stationary"
)

tapping_df = load_challenge_folder(
    DATA_DIR / "tapping",
    "tapping"
)

challenge_df = pd.concat(
    [
        stationary_df,
        tapping_df
    ],
    ignore_index=True
)

In [ ]:
from pathlib import Path

DATA_DIR = Path("../../data/raw")

freq60_low_tables = []

for file_path in sorted(
    (DATA_DIR / "frq_60_low").glob("*.csv")
):
    df = pd.read_csv(file_path)

    feature_df = extract_recording_features(
        df=df,
        recording_id=file_path.stem,
        condition="freq_60_low",
        label=1,
        trim_seconds=1
    )

    freq60_low_tables.append(feature_df)

freq60_low_df = pd.concat(
    freq60_low_tables,
    ignore_index=True
)

In [ ]:
normal40_df = features_df[
    features_df["condition"] == "normal_40"
]

print("NORMAL 40 Hz RMS")
print(
    normal40_df["rms"].describe()
)

print("\nLOW-VOLUME 60 Hz RMS")
print(
    freq60_low_df["rms"].describe()
)

In [ ]:
print(
    "\n40 Hz:",
    normal40_df["rms"].min(),
    "to",
    normal40_df["rms"].max()
)

print(
    "60 Hz low:",
    freq60_low_df["rms"].min(),
    "to",
    freq60_low_df["rms"].max()
)

In [ ]:
print(
    freq60_low_df[
        [
            "recording_id",
            "window",
            "rms",
            "dominant_freq",
            "energy_mid",
            "energy_high"
        ]
    ]
)